# 4.04 Productivity · SaaS Loaders

팀이 문서를 실제로 쌓아두는 곳(Notion, Slack, Confluence, GitHub, Figma)에서 바로 `Document`로 가져오는 로더를 다룬다.

## 학습 목표

- `NotionDirectoryLoader`: Notion export 압축 해제본을 파일시스템에서 읽기
- `SlackDirectoryLoader`: Slack export zip → 채널·스레드 단위 `Document`
- `GithubFileLoader`: public repo는 `GITHUB_TOKEN` 없이, private는 토큰으로
- `ConfluenceLoader`: space/CQL 쿼리 기반 대량 페이지 적재
- `FigmaFileLoader`: 노드 ID 기반 디자인 파일 메타데이터

## 언제 쓰나

- 팀 위키(Notion/Confluence) RAG
- Slack 지식 검색 (특정 채널의 Q&A)
- 오픈소스 저장소 README/docs를 쉽게 훑어보기
- 디자인 시스템 문서 연동

## 4.04.1 환경 설정 · 키 확인

GitHub **공개 저장소**는 토큰 없이 읽을 수 있다(비공개는 필수).
Notion/Slack는 export 파일만 있으면 오프라인 실습 가능.
Confluence/Figma는 API 토큰이 반드시 필요.

In [ ]:
# !pip install -U langchain-community PyGithub atlassian-python-api
import os, json, zipfile
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)

keys = {
    "GITHUB_TOKEN":      bool(os.environ.get("GITHUB_TOKEN")),      # 없어도 공개 repo 가능
    "CONFLUENCE_API_KEY":bool(os.environ.get("CONFLUENCE_API_KEY")),
    "FIGMA_ACCESS_TOKEN":bool(os.environ.get("FIGMA_ACCESS_TOKEN")),
}
print("키 존재 여부:", keys)

SAMPLE_DIR = Path("./_productivity_samples").resolve()
SAMPLE_DIR.mkdir(exist_ok=True)

# Notion export 모사: 마크다운 파일 몇 개가 든 폴더
NOTION_DIR = SAMPLE_DIR / "notion_export"
NOTION_DIR.mkdir(exist_ok=True)
(NOTION_DIR / "Roadmap 2026.md").write_text("# Roadmap 2026\n\n- Q1 런칭\n- Q2 다국어\n", encoding="utf-8")
(NOTION_DIR / "On-call rotation.md").write_text("# On-call\n\nMondays 9am handoff.\n", encoding="utf-8")

# Slack export 모사: 최소 channels.json/users.json + 채널 폴더 + 메시지 JSON
SLACK_DIR = SAMPLE_DIR / "slack_export"
SLACK_DIR.mkdir(exist_ok=True)
(SLACK_DIR / "channels.json").write_text(json.dumps([{"id": "C1", "name": "random"}]), encoding="utf-8")
(SLACK_DIR / "users.json").write_text(json.dumps([{"id": "U1", "profile": {"display_name": "alice"}}]), encoding="utf-8")
random_dir = SLACK_DIR / "random"
random_dir.mkdir(exist_ok=True)
(random_dir / "2026-01-01.json").write_text(json.dumps([
    {"user": "U1", "text": "Happy new year!", "ts": "1735689600.000000"},
    {"user": "U1", "text": "Don't forget the demo on Jan 15.", "ts": "1735776000.000000"},
]), encoding="utf-8")

SLACK_ZIP = SAMPLE_DIR / "slack_export.zip"
with zipfile.ZipFile(SLACK_ZIP, "w") as zf:
    for p in SLACK_DIR.rglob("*"):
        zf.write(p, arcname=p.relative_to(SLACK_DIR))

assert NOTION_DIR.exists() and SLACK_ZIP.exists()
print("samples ready:", [p.name for p in SAMPLE_DIR.iterdir()])

## 4.04.2 `NotionDirectoryLoader` — export 폴더 읽기

Notion에서 `Export > Markdown & CSV`로 받으면 페이지 제목이 파일명인 .md들이 풀린다.
이 로더는 해당 폴더 경로 하나만 받고, 각 .md를 `Document` 1건으로 만든다.
서브페이지는 중첩 폴더로 펼쳐져 있어 `recursive`가 필요 없다(자동 탐색).

제목/속성/날짜 등은 `metadata_func`가 아닌 **파일명 정규식**으로 뽑아 직접 주입하는 것이 일반적이다.

In [ ]:
from langchain_community.document_loaders import NotionDirectoryLoader

notion = NotionDirectoryLoader(str(NOTION_DIR))
docs = notion.load()
print("페이지:", len(docs))
for d in docs:
    print("-", d.metadata.get("source"), "|", d.page_content[:60].replace("\n", " "))

## 4.04.3 `SlackDirectoryLoader` — export zip 파싱

Slack admin이 내보낸 **export zip**을 그대로 받는다. 로더는 내부에서 압축을 풀고, `users.json`/`channels.json`을 참조해 user_id → display_name을 치환한다.

- `workspace_url`: 지정하면 메시지 metadata에 `https://<ws>/archives/<chan>/p<ts>` 퍼머링크 생성
- 스레드의 `thread_ts`도 metadata로 보존

In [ ]:
from langchain_community.document_loaders import SlackDirectoryLoader

slack = SlackDirectoryLoader(str(SLACK_ZIP), workspace_url="https://demo.slack.com")
sdocs = slack.load()
print("메시지 수:", len(sdocs))
for d in sdocs[:3]:
    print("-", d.metadata.get("source"), "| chan:", d.metadata.get("channel"),
          "| user:", d.metadata.get("user"), "| text:", d.page_content[:50])

## 4.04.4 `GithubFileLoader` — 공개 저장소(토큰 없이) vs 비공개(토큰 필요)

LangChain의 `GithubFileLoader`는 **pydantic 모델 기반**이라 키워드 인자만 받는다. 주요 필드:

| 필드 | 설명 |
|------|------|
| `repo` | `owner/name` 형식 |
| `access_token` | PAT 혹은 `GITHUB_TOKEN`. 공개 repo는 빈 문자열 허용(레이트 낮음) |
| `branch` | 기본 `main` |
| `github_api_url` | GH Enterprise는 `https://ghe.example/api/v3` |
| `file_filter` | `lambda path: path.endswith(".md")` 같은 filter |

### 공개 저장소도 토큰이 필요

의 는 pydantic validator가 을 필수로 요구한다. 공개 repo라도 을 먼저 세팅해야 한다. 토큰 없는 공개 repo 적재는  직접 fetch로 대체한다.

In [ ]:
from langchain_community.document_loaders import GithubFileLoader

# 실제 langchain-community 구현은 access_token을 필수로 요구한다.
# 공개 저장소라도 빈 값은 pydantic validator에서 걸림.
# GITHUB_TOKEN 환경변수를 먼저 세팅하고 실행하라.
if keys["GITHUB_TOKEN"]:
    gh_public = GithubFileLoader(
        repo="langchain-ai/langchain",
        access_token=os.environ["GITHUB_TOKEN"],
        branch="master",
        file_filter=lambda p: p == "README.md",
    )
    pub_docs = gh_public.load()
    print("README 수:", len(pub_docs))
    if pub_docs:
        print("source:", pub_docs[0].metadata.get("source"))
        print("chars :", len(pub_docs[0].page_content))
        print("preview:", pub_docs[0].page_content[:150].replace("\n", " "))
else:
    print("GITHUB_TOKEN 미설정 — 공개 repo라도 이 로더는 토큰이 필수다.")
    print("대안: requests.get('https://raw.githubusercontent.com/<org>/<repo>/<branch>/<path>') 로 직접 fetch")
    import requests
    raw = requests.get("https://raw.githubusercontent.com/langchain-ai/langchain/master/README.md", timeout=10)
    print("raw fetch status:", raw.status_code, "| bytes:", len(raw.text))


### 비공개 / 대량 파일 — 토큰 경로

실제 조직 저장소에서는 토큰을 PAT(fine-grained) 또는 GitHub App installation token으로 준다.
레이트 제한이 완화되고 private repo 접근이 가능해진다.

In [ ]:
# assert keys["GITHUB_TOKEN"], "GITHUB_TOKEN 필요"
# gh_private = GithubFileLoader(
#     repo="my-org/internal-docs",
#     access_token=os.environ["GITHUB_TOKEN"],
#     branch="main",
#     file_filter=lambda p: p.endswith((".md", ".mdx")),
# )
# docs = gh_private.load()
# print("사내 문서 수:", len(docs))

print("비공개 저장소 섹션: GITHUB_TOKEN + repo 지정 시 위 주석 해제.")
print("토큰 준비 상태:", keys["GITHUB_TOKEN"])

## 4.04.5 `ConfluenceLoader` — CQL 기반 선택적 수집

Confluence는 **space 단위가 너무 크기 때문에** CQL(Confluence Query Language)로 필터링하는 게 실무 패턴이다.

- `url`: `https://<org>.atlassian.net/wiki`
- `api_key` + `username`: Cloud는 API 토큰 + 이메일
- `space_key` / `page_ids` / `cql`: 범위 지정
- `limit` / `max_pages`: 페이징
- `include_attachments` / `include_comments`: 첨부·댓글까지 가져올지

In [ ]:
# assert keys["CONFLUENCE_API_KEY"], "CONFLUENCE_API_KEY + CONFLUENCE_USER 필요"
# from langchain_community.document_loaders import ConfluenceLoader
# conf = ConfluenceLoader(
#     url="https://myorg.atlassian.net/wiki",
#     username=os.environ["CONFLUENCE_USER"],
#     api_key=os.environ["CONFLUENCE_API_KEY"],
#     cql='space = "ENG" AND type = "page" AND lastmodified > "2026-01-01"',
#     limit=25,
#     max_pages=100,
#     include_attachments=False,
# )
# for d in conf.load()[:3]:
#     print("-", d.metadata.get("title"), "|", len(d.page_content), "chars")

print("Confluence 섹션: CONFLUENCE_USER + CONFLUENCE_API_KEY + URL 필요.")
print("준비 상태:", keys["CONFLUENCE_API_KEY"])

## 4.04.6 `FigmaFileLoader` — 디자인 노드 메타

Figma 로더는 파일 내 **노드 ID 리스트**를 받아 각 노드의 구조를 `Document`로 반환한다.
텍스트가 많은 문서형 파일(디자인 시스템 가이드)에 유용하다.

| 인자 | 설명 |
|------|------|
| `access_token` | Figma personal access token |
| `ids` | 쉼표 구분 노드 ID (`0:1,0:2`) |
| `key` | Figma 파일 키 (URL에서 `file/<key>` 부분) |

In [ ]:
# assert keys["FIGMA_ACCESS_TOKEN"], "FIGMA_ACCESS_TOKEN 필요"
# from langchain_community.document_loaders import FigmaFileLoader
# figma = FigmaFileLoader(
#     access_token=os.environ["FIGMA_ACCESS_TOKEN"],
#     ids=os.environ.get("FIGMA_NODE_IDS", "0:1"),
#     key=os.environ["FIGMA_FILE_KEY"],
# )
# for d in figma.load():
#     print(d.metadata.get("source"), "|", len(d.page_content))

print("Figma 섹션: FIGMA_ACCESS_TOKEN + FIGMA_FILE_KEY + FIGMA_NODE_IDS 필요.")
print("준비 상태:", keys["FIGMA_ACCESS_TOKEN"])

## 4.04.7 split · embed 연동 — Notion + Slack 결합 RAG

Notion 문서와 Slack 메시지를 한 벡터스토어에 같이 넣되, `metadata['source_type']` 필드로 필터링 가능하게 만든다.
retriever 단계에서 `filter={"source_type": "notion"}` 등으로 범위를 줄일 수 있다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

assert os.environ.get("OPENAI_API_KEY"), "Chroma 임베딩용 OPENAI_API_KEY 필요"

# source_type 태깅
for d in docs:
    d.metadata["source_type"] = "notion"
for d in sdocs:
    d.metadata["source_type"] = "slack"

merged = docs + sdocs
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30).split_documents(merged)

store = Chroma.from_documents(chunks, embedding=OpenAIEmbeddings(model="text-embedding-3-small"), collection_name="productivity_demo")

for hit in store.similarity_search("roadmap demo 일정", k=3):
    print("-", hit.metadata.get("source_type"), "|", hit.page_content[:80].replace("\n", " "))

## 체크리스트

- [ ] `GithubFileLoader`는 공개 repo에서도 동작하지만 익명은 시간당 60회 제한
- [ ] Confluence는 space 전체보다 `cql`로 범위를 좁히는 게 안전
- [ ] Notion/Slack은 export 파일만 있으면 오프라인 처리 가능
- [ ] 다중 소스 RAG는 `metadata['source_type']`로 필터링 여지를 남긴다

## 다음

- `05_structured_and_code.ipynb` — CSV/JSON/HTML/Git 로더
- `../05_retrievers/` — 메타데이터 필터 기반 retriever

## 참고

- Notion: https://python.langchain.com/docs/integrations/document_loaders/notion/
- Slack: https://python.langchain.com/docs/integrations/document_loaders/slack/
- GitHub: https://python.langchain.com/docs/integrations/document_loaders/github/
- Confluence: https://python.langchain.com/docs/integrations/document_loaders/confluence/
- Figma: https://python.langchain.com/docs/integrations/document_loaders/figma/